In [19]:
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.prebuilt import ToolNode
from langchain_groq import ChatGroq
from langchain_core.tools import tool


In [20]:
@tool
def square(n: int) -> int:
    """Square of a number"""
    return n*n


In [21]:
@tool
def add(a: float, b: float) -> float:
    """Adds two numbers"""
    return a+b

In [22]:
tools = [square, add]
llm = ChatGroq(model="openai/gpt-oss-120b")
llm_with_tools = llm.bind_tools(tools)

In [23]:
# Agent node
def call_model(state: MessagesState) -> dict:
    response = llm_with_tools.invoke(state['messages'])
    return {
        'messages': response
    }


In [24]:
# routing function
def should_continue(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END


In [25]:
builder = StateGraph(MessagesState)

# node
builder.add_node('agent', call_model)
builder.add_node('tools', ToolNode (tools=tools))

# edge
builder.add_edge(START, "agent")

builder.add_conditional_edges(
    'agent', 
    should_continue,
    {
        'tools': 'tools',
        END: END
    }
)

builder.add_edge('tools', 'agent')

In [26]:
graph = builder.compile()

In [27]:
res = graph.invoke({
    'messages': [{
                'role':'user',
                'content': "what is 9 square, then add 10 to it."
    }]
})

In [30]:
print(res['messages'][-1].content)

The result is **91**.
